# Facility location with PySCIPOpt

Choose which **facilities** to open and how to serve every **customer**, so as
to minimize the total cost: a fixed cost per open facility plus a serving cost
(the euclidean distance) for every customer-to-facility assignment.

The capacitated facility location problem is solved by **Benders' decomposition**:
- a *master* problem decides which facilities to open;
- a *subproblem* assigns customer demand to the open facilities.

Run the cells below, then use the **🧑 Customer / 🏭 Location** toggle to pick
what a click drops, **click on the canvas to add nodes**, and use the
**Solve** / **Reset** buttons.

Example derived from the [PySCIPOpt repository](https://github.com/scipopt/PySCIPOpt) under MIT license.
Copyright (c) by Joao Pedro PEDROSO and Mikio KUBO, 2012

In [ ]:
import pyscipopt as scip
from pyscipopt import SCIP_PARAMSETTING

import flp_utils as utils


def solve_flp(
    I: list[int],
    J: list[int],
    d: dict[int, float],
    M: dict[int, float],
    f: dict[int, float],
    c: dict[tuple[int, int], float],
) -> tuple[float, list[int], list[tuple[int, int]]]:
    """solve_flp -- capacitated facility location via Benders' decomposition
    Parameters:
        - I: set/list of customers
        - J: set/list of candidate facilities
        - d[i]: demand of customer i
        - M[j]: capacity of facility j
        - f[j]: fixed cost of opening facility j
        - c[i, j]: cost of serving customer i from facility j
    Returns the optimum objective value, the open facilities, and the list of
    (customer, facility) assignments.
    """
    master = scip.Model("flp-master")
    subprob = scip.Model("flp-subprob")

    # master problem: choose which facilities to open
    y = {j: master.addVar(vtype="B", name="y(%s)" % j) for j in J}
    master.setObjective(scip.quicksum(f[j] * y[j] for j in J), "minimize")
    master.data = y

    # subproblem: serve every customer's demand from the open facilities.
    # suby mirrors the master's y (matched by name) for Benders' decomposition.
    x, suby = {}, {}
    for j in J:
        suby[j] = subprob.addVar(vtype="B", name="y(%s)" % j)
        for i in I:
            x[i, j] = subprob.addVar(vtype="C", name="x(%s,%s)" % (i, j))

    for i in I:
        subprob.addCons(scip.quicksum(x[i, j] for j in J) == d[i], "Demand(%s)" % i)
    for j in J:
        subprob.addCons(scip.quicksum(x[i, j] for i in I) <= M[j] * suby[j], "Capacity(%s)" % j)
    for (i, j) in x:
        subprob.addCons(x[i, j] <= d[i] * suby[j], "Strong(%s,%s)" % (i, j))

    subprob.setObjective(scip.quicksum(c[i, j] * x[i, j] for (i, j) in x), "minimize")
    subprob.data = x, suby

    # initialize the default Benders' decomposition with the subproblem
    master.setPresolve(SCIP_PARAMSETTING.OFF)
    master.setBoolParam("misc/allowstrongdualreds", False)
    master.setBoolParam("benders/copybenders", False)
    master.hideOutput()
    master.initBendersDefault(subprob)

    master.optimize()
    master.computeBestSolSubproblems()  # solve the subproblem at the best master solution

    EPS = 1.e-6
    open_facilities = [j for j in J if master.getVal(y[j]) > EPS]
    assignments = [(i, j) for (i, j) in x if subprob.getVal(x[i, j]) > EPS]
    obj = master.getObjVal()
    return obj, open_facilities, assignments

In [ ]:
utils.flp_widget(solve_flp)